# Labs 19 and 20: Comps, calculators and chart reads

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
# From the slide: Calculators that leave a trace
import hashlib, json
TRACES = {}
def trace(name, value, inputs, formula):
    t = dict(calc=name, value=value, inputs=inputs,
             formula=formula)
    raw = json.dumps(t, sort_keys=True).encode()
    t["calc_id"] = hashlib.sha1(raw).hexdigest()[:12]
    TRACES[t["calc_id"]] = t
    return t

def ltm(fy, cur, prior):             # three facts
    v = fy["value"] + cur["value"] - prior["value"]
    ids = [f["fact_id"] for f in (fy, cur, prior)]
    return trace("ltm", v, ids, "FY + YTD_cur - YTD_prior")

def ev(price, sh, debt, cash, pref, nci):
    # Every input, including zero-valued preferred stock and NCI, is a fact.
    v = (price["value"] * sh["value"] + debt["value"]
         + pref["value"] + nci["value"] - cash["value"])
    ids = [f["fact_id"] for f in (price, sh, debt, pref, nci, cash)]
    return trace("ev", v, ids, "P*sh+debt+pref+nci-cash")


In [3]:
import pandas as pd
cf = pd.read_csv("data/reference/comps_facts_sample.csv"); px = pd.read_csv("data/reference/prices_asof_20260930.csv").set_index("ticker")
FACTS = {}
COLUMNS = {
    "fy_rev": "fy_revenue", "ytd_rev": "ytd_revenue_current",
    "ytd_rev_prior": "ytd_revenue_prior", "fy_ebitda": "fy_ebitda",
    "ytd_ebitda": "ytd_ebitda_current", "ytd_ebitda_prior": "ytd_ebitda_prior",
    "shares": "shares_out", "debt": "debt", "cash": "cash",
    "preferred": "preferred", "nci": "nci", "price": "close",
}
def fact(t, name, v):
    is_price = name == "price"
    source = "data/reference/prices_asof_20260930.csv" if is_price else "data/reference/comps_facts_sample.csv"
    table = px.reset_index() if is_price else cf
    matches = table.index[table["ticker"] == t].tolist()
    assert len(matches) == 1
    row_index = matches[0]
    column = COLUMNS[name]
    assert float(table.loc[row_index, column]) == float(v)
    f = {
        "fact_id": f"F-{t}-{name}", "value": float(v),
        "tier": 1, "source_type": "fictional_classroom_fixture",
        "tier_note": "Tier-1-like calculator input for this exercise; not a real filed or market fact.",
        "citation": {"kind": "csv_cell", "source_path": source,
                     "row": row_index + 2, "ticker": t, "column": column},
    }
    if is_price:
        f["citation"]["as_of"] = str(px.loc[t, "as_of"])
        f["citation"]["currency"] = str(px.loc[t, "currency"])
    FACTS[f["fact_id"]] = f
    return f
rows = []
for _, r in cf.iterrows():
    t = r["ticker"]
    rev = ltm(fact(t, "fy_rev", r.fy_revenue), fact(t, "ytd_rev", r.ytd_revenue_current), fact(t, "ytd_rev_prior", r.ytd_revenue_prior))
    ebitda = ltm(fact(t, "fy_ebitda", r.fy_ebitda), fact(t, "ytd_ebitda", r.ytd_ebitda_current), fact(t, "ytd_ebitda_prior", r.ytd_ebitda_prior))
    e = ev(fact(t, "price", px.loc[t, "close"]), fact(t, "shares", r.shares_out), fact(t, "debt", r.debt), fact(t, "cash", r.cash), pref=fact(t, "preferred", r.preferred), nci=fact(t, "nci", r.nci))
    evr = trace("ev_revenue", e["value"] / rev["value"], [e["calc_id"], rev["calc_id"]], "EV / LTM revenue")
    eve = trace("ev_ebitda", e["value"] / ebitda["value"], [e["calc_id"], ebitda["calc_id"]], "EV / LTM EBITDA") if ebitda["value"] > 0 else None
    rows.append({"ticker": t, "ltm_revenue": rev["value"], "ltm_revenue_calc_id": rev["calc_id"], "ev": e["value"], "ev_calc_id": e["calc_id"],
                 "ev_revenue": evr["value"], "ev_revenue_calc_id": evr["calc_id"],
                 "ev_ebitda": eve["value"] if eve else None, "ev_ebitda_calc_id": eve["calc_id"] if eve else None})   # n.m. when EBITDA <= 0
comps = pd.DataFrame(rows)
Path("outputs/comps").mkdir(parents=True, exist_ok=True); comps.to_csv("outputs/comps/deal_a.csv", index=False)   # CS2 section 8: a calc_id beside every metric
print(comps[["ticker", "ev_revenue", "ev_revenue_calc_id", "ev_ebitda", "ev_ebitda_calc_id"]].round(2).to_string(index=False))
print(comps[["ev_revenue", "ev_ebitda"]].quantile([.25, .5, .75]).round(2))   # n.m. (negative EBITDA) excluded

ticker  ev_revenue ev_revenue_calc_id  ev_ebitda ev_ebitda_calc_id
  SMPA        3.21       1f53c446513a      16.26      afc9f4a34dd8
  SMPB        2.63       401bd14d3f0d      19.25      4addb6b168ab
  SMPC        2.78       d1272cf1e4bb      12.77      dcdc8105a4d8
  SMPD        2.93       54fb1aeb3afb        NaN               NaN
  SMPE        4.01       4c7d3a3bdb88      19.80      3c11e070a2d6
  SMPF        3.09       4c3421af0f71      19.60      3ca84721b262
      ev_revenue  ev_ebitda
0.25        2.81      16.26
0.50        3.01      19.25
0.75        3.18      19.60


In [4]:
# Chart vs table: a recorded chart read against the seller's table
reads = json.load(open("data/dataroom/deal_a/chart_reads.json"))
F = {(f["metric"], f["period"]): f["value"] for f in sd.seller_facts()}
for c in reads:
    gap = (c["value"] - F[(c["metric"], c["period"])]) / F[(c["metric"], c["period"])]
    print(c["chart"], f"chart {c['value']/1e6:.1f}m vs table {F[(c['metric'], c['period'])]/1e6:.1f}m, gap {gap:.1%}",
          "-> DISCREPANCY" if abs(gap) > llm.P["reconcile"]["tolerance_rel"] else "-> ok")

Revenue by fiscal year (CIM, page 4) chart 42.0m vs table 41.2m, gap 1.9% -> DISCREPANCY


**Exercise 19/20.** Trace SMPA's EV/EBITDA back to its input facts with the calc ids. Why is SMPD's EV/EBITDA left out of the quartiles?

In [5]:
# 1. Retrieve SMPA's input data.
r = cf.set_index("ticker").loc["SMPA"]
price = fact("SMPA", "price", px.loc["SMPA", "close"])

# 2. Calculate LTM EBITDA.
ebitda_trace = ltm(
    fact("SMPA", "fy_ebitda", r.fy_ebitda),
    fact("SMPA", "ytd_ebitda", r.ytd_ebitda_current),
    fact("SMPA", "ytd_ebitda_prior", r.ytd_ebitda_prior),
)

# 3. Calculate enterprise value.
ev_trace = ev(
    price,
    fact("SMPA", "shares", r.shares_out),
    fact("SMPA", "debt", r.debt),
    fact("SMPA", "cash", r.cash),
    pref=fact("SMPA", "preferred", r.preferred),
    nci=fact("SMPA", "nci", r.nci),
)

# 4. Connect EV/EBITDA to both calculation IDs.
multiple_trace = trace(
    "ev_ebitda",
    ev_trace["value"] / ebitda_trace["value"],
    [ev_trace["calc_id"], ebitda_trace["calc_id"]],
    "EV / LTM EBITDA",
)

for result in [ebitda_trace, ev_trace, multiple_trace]:
    print(json.dumps(result, indent=2))

# Verify this matches the calculation already in the comps table.
saved_id = comps.loc[
    comps["ticker"] == "SMPA", "ev_ebitda_calc_id"
].iloc[0]
assert multiple_trace["calc_id"] == saved_id

# 5. Inspect SMPD's EBITDA.
d = cf.set_index("ticker").loc["SMPD"]
smpd_ebitda = (
    d.fy_ebitda + d.ytd_ebitda_current - d.ytd_ebitda_prior
)

print(f"\nSMPA EV/EBITDA: {multiple_trace['value']:.2f}x")
print(f"SMPD LTM EBITDA: ${smpd_ebitda:,.0f}")
# Walk from the table's multiple calc_id to every cited input.
def walk(reference, depth=0):
    prefix = "  " * depth
    if reference in FACTS:
        f = FACTS[reference]
        print(prefix + f"{reference} = {f['value']:,.2f} | tier {f['tier']} (fictional sample) | {json.dumps(f['citation'])}")
        return {reference}
    if reference not in TRACES:
        raise KeyError(f"Unresolved trace reference: {reference}")
    calculation = TRACES[reference]
    print(prefix + f"{calculation['calc']} = {calculation['value']:,.2f} [{calculation['formula']}] ({reference})")
    leaves = set()
    for child in calculation["inputs"]:
        leaves.update(walk(child, depth + 1))
    return leaves

print("\nFull SMPA EV/EBITDA trace (all sources are fictional classroom CSVs):")
leaves = walk(saved_id)
assert leaves == {f"F-SMPA-{name}" for name in (
    "price", "shares", "debt", "preferred", "nci", "cash",
    "fy_ebitda", "ytd_ebitda", "ytd_ebitda_prior",
)}

# Re-read source files independently and verify every cited leaf.
import csv, math
for fid in leaves:
    f = FACTS[fid]
    citation = f["citation"]
    with open(citation["source_path"], newline="") as source_file:
        source_rows = list(csv.DictReader(source_file))
    source_row = source_rows[citation["row"] - 2]
    assert source_row["ticker"] == citation["ticker"]
    assert float(source_row[citation["column"]]) == f["value"]

# Arithmetic cross-check from the cited source values, independent of trace functions.
assert math.isclose(ev_trace["value"], 42.10 * 61_000_000 + 300_000_000 - 120_000_000)
assert ebitda_trace["value"] == 160_000_000 + 88_000_000 - 79_000_000
assert math.isclose(multiple_trace["value"], 2_748_100_000 / 169_000_000)
assert smpd_ebitda == -1_000_000
assert pd.isna(comps.set_index("ticker").loc["SMPD", "ev_ebitda"])
assert comps["ev_ebitda"].count() == 5

# Make the calculation IDs in the saved CSV resolvable outside this notebook.
for calc in TRACES.values():
    assert all(ref in TRACES or ref in FACTS for ref in calc["inputs"])
trace_path = Path("outputs/comps/deal_a_traces.json")
trace_path.write_text(json.dumps({"facts": FACTS, "calculations": TRACES}, indent=2) + "\n")
print("All nine SMPA leaf citations verified; saved", trace_path)


{
  "calc": "ltm",
  "value": 169000000.0,
  "inputs": [
    "F-SMPA-fy_ebitda",
    "F-SMPA-ytd_ebitda",
    "F-SMPA-ytd_ebitda_prior"
  ],
  "formula": "FY + YTD_cur - YTD_prior",
  "calc_id": "d68c65f5ec09"
}
{
  "calc": "ev",
  "value": 2748100000.0,
  "inputs": [
    "F-SMPA-price",
    "F-SMPA-shares",
    "F-SMPA-debt",
    "F-SMPA-preferred",
    "F-SMPA-nci",
    "F-SMPA-cash"
  ],
  "formula": "P*sh+debt+pref+nci-cash",
  "calc_id": "3dbacaf9f994"
}
{
  "calc": "ev_ebitda",
  "value": 16.26094674556213,
  "inputs": [
    "3dbacaf9f994",
    "d68c65f5ec09"
  ],
  "formula": "EV / LTM EBITDA",
  "calc_id": "afc9f4a34dd8"
}

SMPA EV/EBITDA: 16.26x
SMPD LTM EBITDA: $-1,000,000

Full SMPA EV/EBITDA trace (all sources are fictional classroom CSVs):
ev_ebitda = 16.26 [EV / LTM EBITDA] (afc9f4a34dd8)
  ev = 2,748,100,000.00 [P*sh+debt+pref+nci-cash] (3dbacaf9f994)
    F-SMPA-price = 42.10 | tier 1 (fictional sample) | {"kind": "csv_cell", "source_path": "data/reference/prices_asof_20

**Trace result.** SMPA's EV/EBITDA is approximately **16.26×**: EV $2,748.1 million divided by LTM EBITDA $169 million. The registered trace now reaches all nine input facts, including the $42.10 price snapshot dated 2026-09-30, shares, debt, preferred stock, NCI, cash, and three EBITDA inputs. Every leaf prints its fact ID, sample tier, CSV file, row, and column. The notebook independently re-reads each cited cell to verify its value. These are fictional classroom sources; this is not a manual verification against real SEC filings.

The CSV and `deal_a_traces.json` are saved together so every calculation ID resolves to a calculation or cited input. EV and multiple calculation IDs change because their input lists now include previously omitted facts; the financial values do not change.

**Why exclude SMPD?** LTM EBITDA is −$4m + $1m − (−$2m) = **−$1m**. EV/EBITDA is therefore not meaningful and is stored as missing. Quartiles use the five remaining multiples, rather than treating SMPD as zero.

**Where can sign errors hide?** Cash is subtracted from EV; debt, preferred stock and NCI are added. Subtracting a negative prior-period EBITDA adds its absolute value. Check whether source liabilities are reported with a negative sign before mapping them into the calculator. Zero preferred stock and NCI still need cited input records rather than being silently assumed.
